In [ ]:
import json
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

[CLINIC150.json](https://www.kaggle.com/datasets/hongtrung/clinc150-dataset) <span style="color: gray"># Kaggle Datase</span>

In [25]:
with open("../../data/CLINIC150.json", 'r') as f:
    clinic_data = json.load(f)

all_data = clinic_data['train'] + clinic_data['val'] + clinic_data['test']
df = pd.DataFrame(all_data, columns=['text', 'intent'])
df.head()

,text,intent
0,what expression would i use to say i love you ...,translate
1,can you tell me how to say 'i do not speak muc...,translate
2,"what is the equivalent of, 'life is good' in f...",translate
3,"tell me how to say, 'it is a beautiful morning...",translate
4,"if i were mongolian, how would i say that i am...",translate


In [26]:
label_mapping = {
    # Todos
    "todo_list": "todo",
    "todo_list_update": "todo",
    "shopping_list": "todo",
    "shopping_list_update": "todo",
    
    # Events
    "schedule_meeting": "event",
    "calendar": "event",
    "calendar_update": "event",
    
    # Reminders
    "reminder": "reminder",
    "reminder_update": "reminder",
    "set_alarm": "reminder",
    
    # Tasks 
    "pay_bill": "task",
    "book_flight": "task",
    "cancel_reservation": "task",
    "auto_pay": "task"
}
df['label'] = df['intent'].map(label_mapping)
df = df.dropna(subset=['label'])
df.head()

,text,intent,label
2000,delete fries from shopping list,shopping_list_update,todo
2001,take off fries from the shopping list,shopping_list_update,todo
2002,please take away the fries from the shopping list,shopping_list_update,todo
2003,remove fries from my shopping list,shopping_list_update,todo
2004,we no longer need fries on the shopping lisr,shopping_list_update,todo


In [27]:
df = df[['text', 'label']]
df['label'].value_counts()

label
todo        600
task        450
event       450
reminder    300
Name: count, dtype: int64

In [28]:
df_train, df_test = train_test_split(df, random_state=42, test_size=0.2)

In [29]:
training_texts = df_train['text'].tolist()
training_labels = df_train['label'].tolist()

classifier = make_pipeline(TfidfVectorizer(), LinearSVC())
classifier.fit(training_texts, training_labels)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('linearsvc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U8](4,)","['event','reminder','task','todo']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


In [30]:
test_texts = df_test['text'].tolist()
test_labels = df_test['label'].tolist()

predictions = classifier.predict(test_texts)

accuracy = accuracy_score(test_labels, predictions)
print(f"Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(test_labels, predictions))

Accuracy: 0.9833

Classification Report:
              precision    recall  f1-score   support

       event       0.99      0.98      0.98        87
    reminder       0.99      0.96      0.97        78
        task       0.99      1.00      0.99        79
        todo       0.97      0.99      0.98       116

    accuracy                           0.98       360
   macro avg       0.98      0.98      0.98       360
weighted avg       0.98      0.98      0.98       360



In [32]:
joblib.dump(classifier, '../../model/intent_classifier.joblib')

['../../model/intent_classifier.joblib']